# TakeMeter — r/TrueFilm

This notebook:

1. Uploads the reviewed `labeled_data_final.csv`
2. Creates a stratified 70/15/15 train/validation/test split
3. Runs the required Groq zero-shot baseline
4. Fine-tunes `distilbert-base-uncased`
5. Evaluates the fine-tuned model
6. Prints confusion matrices and misclassified examples
7. Saves evaluation outputs for the README

Final labels:

- `analysis`
- `opinion`
- `discussion_question`


## 1. Install dependencies

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn evaluate groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 8.5 MB/s eta 0:00:00


## 2. Imports

In [ ]:
import json
import pandas as pd
import numpy as np

from google.colab import files, userdata
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from groq import Groq


## 3. Upload the dataset

Upload **`labeled_data_final.csv`** when prompted.


In [ ]:
uploaded = files.upload()


Saving labeled_data_final.csv to labeled_data_final.csv


In [ ]:
DATASET_FILE = "labeled_data_final.csv"

df = pd.read_csv(DATASET_FILE)

print("Shape:", df.shape)
print("\nLabel distribution:")
print(df["label"].value_counts())

df.head()


Shape: (225, 7)

Label distribution:
label
analysis               94
discussion_question    86
opinion                45
Name: count, dtype: int64


,text,label,notes,source_url,source_type,qc_status,qc_note
0,Shutter Island is a masterclass in using bad f...,analysis,Reasoning or interpretation is the main substa...,https://www.reddit.com/r/TrueFilm/comments/1pw...,post,reviewed,Kept original label after QC/spot-check
1,Notes from One Battle After Another\n\nWhat fo...,analysis,Reasoning or interpretation is the main substa...,https://www.reddit.com/r/TrueFilm/comments/1nr...,post,reviewed,Kept original label after QC/spot-check
2,One Battle After Another has one of the finest...,analysis,Reasoning or interpretation is the main substa...,https://www.reddit.com/r/TrueFilm/comments/1nr...,post,reviewed,Kept original label after QC/spot-check
3,So I watched Seven Samurai and had a few thoug...,analysis,Reasoning or interpretation is the main substa...,https://www.reddit.com/r/TrueFilm/comments/1lk...,post,reviewed,Kept original label after QC/spot-check
4,"Boiling Point, the most ""Kitano"" of Kitano's f...",analysis,Reasoning or interpretation is the main substa...,https://www.reddit.com/r/TrueFilm/comments/1no...,post,reviewed,Kept original label after QC/spot-check


## 4. Keep only the columns needed for training

In [ ]:
df = df[["text", "label"]].dropna().copy()

label2id = {
    "analysis": 0,
    "opinion": 1,
    "discussion_question": 2
}

id2label = {
    0: "analysis",
    1: "opinion",
    2: "discussion_question"
}

df["label_id"] = df["label"].map(label2id)

assert df["label_id"].notna().all(), "Unexpected label found in dataset."

df.head()


,text,label,label_id
0,Shutter Island is a masterclass in using bad f...,analysis,0
1,Notes from One Battle After Another\n\nWhat fo...,analysis,0
2,One Battle After Another has one of the finest...,analysis,0
3,So I watched Seven Samurai and had a few thoug...,analysis,0
4,"Boiling Point, the most ""Kitano"" of Kitano's f...",analysis,0


## 5. Create the 70 / 15 / 15 split

The split is stratified so each set keeps roughly the same class proportions.

After this point, treat the **test set as locked**.


In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nTRAIN DISTRIBUTION")
print(train_df["label"].value_counts())

print("\nVALIDATION DISTRIBUTION")
print(val_df["label"].value_counts())

print("\nTEST DISTRIBUTION")
print(test_df["label"].value_counts())


Train: 157
Validation: 34
Test: 34

TRAIN DISTRIBUTION
label
analysis               66
discussion_question    60
opinion                31
Name: count, dtype: int64

VALIDATION DISTRIBUTION
label
analysis               14
discussion_question    13
opinion                 7
Name: count, dtype: int64

TEST DISTRIBUTION
label
analysis               14
discussion_question    13
opinion                 7
Name: count, dtype: int64


# Zero-shot baseline

The assignment requires the Groq model:

`meta-llama/llama-4-scout-17b-16e-instruct`

Before running the next cell:

1. Open the **Secrets** panel in Colab (key icon in the left sidebar)
2. Add a secret named `GROQ_API_KEY`
3. Enable notebook access


In [ ]:
client = Groq(
    api_key=userdata.get("GROQ_API_KEY")
)

print("Groq client ready.")


Groq client ready.


## 6. Baseline classification function

In [ ]:
def baseline_classify(text):
    prompt = f"""
You are classifying discourse from the r/TrueFilm Reddit community.

Choose exactly ONE label:

analysis:
The main purpose is to explain, interpret, or argue something about film
using meaningful reasoning, examples, observations, themes, filmmaking
techniques, narrative structure, historical context, or comparisons.

opinion:
The main purpose is expressing a judgment, preference, or evaluation about
a movie, filmmaker, actor, performance, genre, or filmmaking choice without
substantially developing the reasoning.

discussion_question:
The primary purpose is asking the community to explain, interpret, compare,
recommend, or debate something related to film.

Decision rules:

If reasoning or interpretation is the main substance, choose analysis.

If the judgment itself is the main substance and reasoning is minimal,
choose opinion.

If the primary goal is getting responses from others, choose
discussion_question, even if the author includes their own opinion.

Return ONLY one of these exact strings:

analysis
opinion
discussion_question

Text:

{text}
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip().lower()


## 7. Test the baseline on one example

In [ ]:
sample_text = test_df.iloc[0]["text"]

print("TEXT:")
print(sample_text)
print("\nPrediction:", baseline_classify(sample_text))
print("Actual:", test_df.iloc[0]["label"])


TEXT:
I love PTA but didn't really think his latest film is all that is being made

Just saw One Battle After Another and even though the cinematography is masterful, as I would expect from PTA, I feel that if they had toned down the actions/chase scenes, the trademark DiCaprio quips, and instead fleshed out a bit more of Benicio's whole thing and let the storylines "brew" a bit more it could have been a way better film. This is where I think Licorice Pizza absolutely kills it—you don't have that many events to play with, so the rhythm of the film allows you to breathe in the themes much more. This is what Richard Linklater does to perfection. And even going back on PTA's work, for example There Will Be Blood, I feel this works way better because as a viewer you are confronting the ugly reality of the story in real time. I didn't get that with this latest one. I also think immigration is one of those topics that are so touchy right now, that there's no way in hell a film critic is gonn

## 8. Run the baseline on the full test set

In [ ]:
baseline_predictions = []

for n, (_, row) in enumerate(test_df.iterrows(), start=1):
    pred = baseline_classify(row["text"])

    if pred not in label2id:
        print(f"Unexpected output on example {n}: {pred}")

    baseline_predictions.append(pred)

    if n % 5 == 0:
        print(f"Completed {n}/{len(test_df)}")


Completed 5/34
Completed 10/34
Completed 15/34
Completed 20/34
Completed 25/34
Completed 30/34


## 9. Baseline metrics

In [ ]:
baseline_true = test_df["label"].tolist()
baseline_pred = baseline_predictions

baseline_accuracy = accuracy_score(
    baseline_true,
    baseline_pred
)

print("Baseline Accuracy:", round(baseline_accuracy, 4))
print()

print(
    classification_report(
        baseline_true,
        baseline_pred,
        labels=[
            "analysis",
            "opinion",
            "discussion_question"
        ],
        zero_division=0
    )
)

baseline_cm = confusion_matrix(
    baseline_true,
    baseline_pred,
    labels=[
        "analysis",
        "opinion",
        "discussion_question"
    ]
)

baseline_cm_df = pd.DataFrame(
    baseline_cm,
    index=[
        "true_analysis",
        "true_opinion",
        "true_discussion_question"
    ],
    columns=[
        "pred_analysis",
        "pred_opinion",
        "pred_discussion_question"
    ]
)

baseline_cm_df


Baseline Accuracy: 0.8824

                     precision    recall  f1-score   support

           analysis       1.00      0.79      0.88        14
            opinion       0.75      0.86      0.80         7
discussion_question       0.87      1.00      0.93        13

           accuracy                           0.88        34
          macro avg       0.87      0.88      0.87        34
       weighted avg       0.90      0.88      0.88        34



,pred_analysis,pred_opinion,pred_discussion_question
true_analysis,11,2,1
true_opinion,0,6,1
true_discussion_question,0,0,13


# Fine-tune DistilBERT

## 10. Convert Pandas splits to Hugging Face datasets

In [ ]:
train_dataset = Dataset.from_pandas(
    train_df[["text", "label_id"]]
)

val_dataset = Dataset.from_pandas(
    val_df[["text", "label_id"]]
)

test_dataset = Dataset.from_pandas(
    test_df[["text", "label_id"]]
)

train_dataset = train_dataset.rename_column("label_id", "labels")
val_dataset = val_dataset.rename_column("label_id", "labels")
test_dataset = test_dataset.rename_column("label_id", "labels")


## 11. Load tokenizer

In [ ]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/157 [00:00<?, ? examples/s]

Map:   0%|          | 0/34 [00:00<?, ? examples/s]

Map:   0%|          | 0/34 [00:00<?, ? examples/s]

## 12. Load the model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    label2id=label2id,
    id2label=id2label
)


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 13. Metrics function

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1
    }


## 14. Training configuration

Starting hyperparameters:

- Epochs: **3**
- Learning rate: **2e-5**
- Batch size: **16**


In [ ]:
training_args = TrainingArguments(
    output_dir="./takemeter_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)


## 15. Create Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)


## 16. Fine-tune

In [ ]:
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1
1,No log,1.006644,0.441176,0.474747,0.358974,0.246201
2,No log,0.964002,0.617647,0.406566,0.512821,0.445926
3,No log,0.947523,0.588235,0.394444,0.487179,0.419527


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=30, training_loss=0.9695274353027343, metrics={'train_runtime': 46.8159, 'train_samples_per_second': 10.061, 'train_steps_per_second': 0.641, 'total_flos': 62393257442304.0, 'train_loss': 0.9695274353027343, 'epoch': 3.0})

## 17. Evaluate on the locked test set

In [ ]:
pred_output = trainer.predict(test_dataset)

predictions = np.argmax(
    pred_output.predictions,
    axis=-1
)

true_labels = pred_output.label_ids

fine_tuned_pred = [
    id2label[int(x)]
    for x in predictions
]

fine_tuned_true = [
    id2label[int(x)]
    for x in true_labels
]

fine_tuned_accuracy = accuracy_score(
    fine_tuned_true,
    fine_tuned_pred
)

print("Fine-Tuned Accuracy:", round(fine_tuned_accuracy, 4))
print()

print(
    classification_report(
        fine_tuned_true,
        fine_tuned_pred,
        labels=[
            "analysis",
            "opinion",
            "discussion_question"
        ],
        zero_division=0
    )
)


Fine-Tuned Accuracy: 0.5882

                     precision    recall  f1-score   support

           analysis       0.61      1.00      0.76        14
            opinion       0.00      0.00      0.00         7
discussion_question       0.55      0.46      0.50        13

           accuracy                           0.59        34
          macro avg       0.38      0.49      0.42        34
       weighted avg       0.46      0.59      0.50        34



## 18. Fine-tuned confusion matrix

In [ ]:
fine_tuned_cm = confusion_matrix(
    fine_tuned_true,
    fine_tuned_pred,
    labels=[
        "analysis",
        "opinion",
        "discussion_question"
    ]
)

fine_tuned_cm_df = pd.DataFrame(
    fine_tuned_cm,
    index=[
        "true_analysis",
        "true_opinion",
        "true_discussion_question"
    ],
    columns=[
        "pred_analysis",
        "pred_opinion",
        "pred_discussion_question"
    ]
)

fine_tuned_cm_df


,pred_analysis,pred_opinion,pred_discussion_question
true_analysis,14,0,0
true_opinion,2,0,5
true_discussion_question,7,0,6


## 19. Inspect wrong predictions

In [ ]:
results_df = test_df[["text", "label"]].copy().reset_index(drop=True)

results_df["baseline_prediction"] = baseline_pred
results_df["fine_tuned_prediction"] = fine_tuned_pred

wrong_df = results_df[
    results_df["label"] != results_df["fine_tuned_prediction"]
].copy()

print("Fine-tuned mistakes:", len(wrong_df))
wrong_df


Fine-tuned mistakes: 14


,text,label,baseline_prediction,fine_tuned_prediction
1,What is a Cinephile?\n\nWhat do you guys consi...,discussion_question,discussion_question,analysis
3,Dangerous Liaisons (1988) and Valmont (1989). ...,opinion,opinion,analysis
4,A short video that channels David Lynch’s exis...,discussion_question,discussion_question,analysis
7,"Yeah, it's an absolutely gorgeous and effectiv...",opinion,opinion,discussion_question
8,"Lack of Feature Medical Dramas, some questions...",discussion_question,discussion_question,analysis
13,How did you interpret one battle after another...,discussion_question,discussion_question,analysis
15,Are you me? The Matrix trilogy really got me o...,opinion,opinion,analysis
16,"The Hero was the film that got me into films, ...",opinion,opinion,discussion_question
17,"Michael Cimino, auteurism, and filmmaker biogr...",discussion_question,discussion_question,analysis
18,"I also agree with you, hence why I stated I st...",opinion,opinion,discussion_question


## 20. Save evaluation outputs

In [ ]:
baseline_report = classification_report(
    baseline_true,
    baseline_pred,
    labels=[
        "analysis",
        "opinion",
        "discussion_question"
    ],
    output_dict=True,
    zero_division=0
)

fine_tuned_report = classification_report(
    fine_tuned_true,
    fine_tuned_pred,
    labels=[
        "analysis",
        "opinion",
        "discussion_question"
    ],
    output_dict=True,
    zero_division=0
)

evaluation_results = {
    "baseline": {
        "model": "openai/gpt-oss-120b",
        "accuracy": baseline_accuracy,
        "classification_report": baseline_report,
        "confusion_matrix": baseline_cm.tolist()
    },
    "fine_tuned": {
        "model": "distilbert-base-uncased",
        "accuracy": fine_tuned_accuracy,
        "classification_report": fine_tuned_report,
        "confusion_matrix": fine_tuned_cm.tolist()
    }
}

with open("evaluation_results.json", "w") as f:
    json.dump(evaluation_results, f, indent=2)

wrong_df.to_csv(
    "wrong_predictions.csv",
    index=False
)

print("Saved:")
print("- evaluation_results.json")
print("- wrong_predictions.csv")


Saved:
- evaluation_results.json
- wrong_predictions.csv


## 21. Download outputs

In [ ]:
files.download("evaluation_results.json")
files.download("wrong_predictions.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 22. Export sample classifications with confidence

Run this cell after the fine-tuned test prediction cell. It computes softmax confidence, exports 5 sample classifications, and saves the fine-tuned confusion matrix image required for the repo.


In [ ]:
import matplotlib.pyplot as plt

# Convert logits into probabilities.
logits = pred_output.predictions
exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
probabilities = exp_logits / exp_logits.sum(axis=1, keepdims=True)
confidences = probabilities.max(axis=1)

sample_results = test_df[["text", "label"]].reset_index(drop=True).copy()
sample_results["predicted_label"] = fine_tuned_pred
sample_results["confidence"] = confidences

# Pick 5 deterministic examples, including correct and incorrect predictions where possible.
correct_rows = sample_results[
    sample_results["label"] == sample_results["predicted_label"]
].head(3)
incorrect_rows = sample_results[
    sample_results["label"] != sample_results["predicted_label"]
].head(2)

sample_classifications = pd.concat(
    [correct_rows, incorrect_rows],
    ignore_index=True
)

sample_classifications.to_csv(
    "sample_classifications.csv",
    index=False
)

# Save confusion matrix image.
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(fine_tuned_cm)

ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(["analysis", "opinion", "discussion_question"], rotation=30, ha="right")
ax.set_yticklabels(["analysis", "opinion", "discussion_question"])
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("Fine-Tuned DistilBERT Confusion Matrix")

for i in range(3):
    for j in range(3):
        ax.text(j, i, str(fine_tuned_cm[i, j]), ha="center", va="center")

fig.tight_layout()
fig.savefig("confusion_matrix.png", dpi=180, bbox_inches="tight")
plt.show()

sample_classifications
